In [6]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Border, Side, Alignment
)
from openpyxl.utils import get_column_letter

In [7]:
# ==========================================
# 1. DATOS DE EJEMPLO
# ==========================================

vecinos = [
    {
        "departamento": "101",
        "nombre": "Juan Pérez",
        "deuda": {
            "jun-26": 261.71,
            "jul-26": 243.33,
            "ago-26": 226.72,
            "sep-26": 208.25
        }
    },
    {
        "departamento": "102",
        "nombre": "María López",
        "deuda": {
            "mar-26": 192.27,
            "sep-26": 200.42
        }
    },
    {
        "departamento": "103",
        "nombre": "Carlos Ruiz",
        "deuda": {
            "nov-25": 180.00,
            "dic-25": 180.00,
            "ene-26": 190.00
        }
    }
]

# ==========================================
# 2. CREAR EL ARCHIVO EXCEL
# ==========================================

wb = Workbook()
ws = wb.active
ws.title = "Mora"

# Colores
color_titulo = "D9EAF7"
color_encabezado = "B4C7E7"
color_total = "E2F0D9"
color_borde = "808080"

# Bordes
borde = Border(
    left=Side(style="thin", color=color_borde),
    right=Side(style="thin", color=color_borde),
    top=Side(style="thin", color=color_borde),
    bottom=Side(style="thin", color=color_borde)
)

# ==========================================
# 3. RECORRER LOS VECINOS
# ==========================================

fila = 1

for vecino in vecinos:

    # Obtener los meses adeudados
    deuda = vecino["deuda"]
    meses = list(deuda.keys())

    # Crear DataFrame
    datos = {
        "Concepto": ["Expensas"],
        **{
            mes: [monto]
            for mes, monto in deuda.items()
        }
    }

    df = pd.DataFrame(datos)

    # Calcular total
    df["Total"] = df[meses].sum(axis=1)

    # --------------------------------------
    # TÍTULO DEL VECINO
    # --------------------------------------

    ws.cell(
        row=fila,
        column=1,
        value=(
            f"Departamento {vecino['departamento']} - "
            f"{vecino['nombre']}"
        )
    )

    # Dar formato al título
    ultima_columna = len(df.columns)

    ws.merge_cells(
        start_row=fila,
        start_column=1,
        end_row=fila,
        end_column=ultima_columna
    )

    celda = ws.cell(fila, 1)
    celda.font = Font(
        bold=True, size=12, color="17365D"
    )
    celda.fill = PatternFill(
        "solid", fgColor=color_titulo
    )
    celda.alignment = Alignment(
        horizontal="left", vertical="center"
    )

    ws.row_dimensions[fila].height = 22

    fila += 1

    # --------------------------------------
    # ENCABEZADOS DE LA TABLA
    # --------------------------------------

    fila_encabezado = fila

    for columna, nombre_columna in enumerate(
        df.columns, start=1
    ):
        celda = ws.cell(
            row=fila,
            column=columna,
            value=nombre_columna
        )

        celda.font = Font(bold=True)
        celda.fill = PatternFill(
            "solid", fgColor=color_encabezado
        )
        celda.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )
        celda.border = borde

    fila += 1

    # --------------------------------------
    # DATOS DE LA TABLA
    # --------------------------------------

    for columna, valor in enumerate(
        df.iloc[0], start=1
    ):
        celda = ws.cell(
            row=fila,
            column=columna,
            value=valor
        )

        celda.border = borde

        if columna == 1:
            celda.alignment = Alignment(
                horizontal="left"
            )
        else:
            celda.alignment = Alignment(
                horizontal="right"
            )
            celda.number_format = '#,##0.00'

    # Resaltar la celda del total
    celda_total = ws.cell(
        row=fila,
        column=ultima_columna
    )

    celda_total.font = Font(bold=True)
    celda_total.fill = PatternFill(
        "solid", fgColor=color_total
    )

    # Siguiente vecino: dejar una fila vacía
    fila += 2

# ==========================================
# 4. AJUSTAR ANCHO DE COLUMNAS
# ==========================================

for columna in range(1, ws.max_column + 1):

    letra = get_column_letter(columna)
    ancho_maximo = 0

    for celda in ws[letra]:
        if celda.value is not None:
            ancho_maximo = max(
                ancho_maximo,
                len(str(celda.value))
            )

    ws.column_dimensions[letra].width = min(
        max(ancho_maximo + 3, 12), 35
    )

# ==========================================
# 5. CONFIGURAR IMPRESIÓN
# ==========================================

ws.sheet_properties.pageSetUpPr.fitToPage = True
ws.page_setup.fitToWidth = 1
ws.page_setup.fitToHeight = 0
ws.page_setup.orientation = "landscape"
ws.page_setup.paperSize = ws.PAPERSIZE_A4

ws.page_margins.left = 0.3
ws.page_margins.right = 0.3
ws.page_margins.top = 0.5
ws.page_margins.bottom = 0.5

# ==========================================
# 6. GUARDAR EL ARCHIVO
# ==========================================

wb.save("reporte_mora.xlsx")

print("Excel generado correctamente.")


Excel generado correctamente.
